# Nebula V-D2 · 1B coding foundation lab

Exactly **1,007,334,912 parameters**. **Prepared, not trained.** Default `TRAINER='two-t4'` is an experimental FULL_SHARD path for two 16GB T4s: FP16, checkpointed activations, microbatch one per GPU, split optimizer backups. It needs a real GPU memory/save/resume test. This notebook has not been run on T4s by the assistant.

A two-update save smoke run precedes the main continuation, which resumes that checkpoint. A single >=40GB GPU trainer remains optional. Both need ample host RAM, private storage and a reviewed coding corpus. GPUs do not remove these requirements. Nothing is purchased or deployed.

Old repositories and models remain untouched. Artifacts stay in **RAM → NEW PRIVATE Hugging Face storage**, never local data/checkpoint files. Private storage quotas still apply. Code/packages use temporary disk. Default phase inventories source manifests; it does not auto-approve unknown data, consume teacher APIs or promote a website model.

1. Run setup/inventory on CPU. Review data provenance/licenses and existing validation splits. Known repositories contain stories, private/consent-bound samples and evaluations as well as useful text; do not import them blindly.
2. Add reviewed copies of candidate entries to `APPROVED_SOURCES`, with `approved=True`, an accurate origin and license. Exclude private chats/review feeds. `PHASE='prepare'` creates the private corpus/tokenizer. It requires at least 5,000 distinct code documents (an admission floor, not enough to promise a competent 1B coder).
3. Enable two T4s, set `PHASE='train'`. Successful smoke-save is followed automatically by resume for the remaining time budget. Default 6 hours; reserve 15 minutes for checkpointing. Sudden kills can still lose work after the last confirmed backup. Quota cannot be detected automatically.
4. These are **foundation coding weights**, not a certified security assistant. Add verified coding-task/repair SFT and independent sandboxed tests before serving. Never execute generated fixes on production.

Inventory covers the two known HF repositories; unconfigured GCS/S3 buckets and inaccessible repos remain uninspected. No claim that all data everywhere has already been moved is made.

## Read-only data audit update
The default audit phase inventories sources, fixes holdout classification, and checks unique JSONL objects within a 512 MB download budget. It prints code/docs/general counts without printing source text. Encoded token corpora are marked uninspected. No source is automatically approved and no training or remote writes occur. Public holdouts stay validation even when an old entry incorrectly says train.


In [ ]:
import sys,subprocess,tempfile,os,json
from pathlib import Path
packages=Path(tempfile.gettempdir())/'nebula-vd2-packages'
subprocess.run([sys.executable,'-m','pip','install','-q','--target',str(packages),'huggingface_hub>=0.30,<2','tokenizers==0.22.2','requests>=2.32,<3','numpy>=1.26,<3'],check=True)
sys.path.insert(0,str(packages))
# Use the GPU provider's supported PyTorch image. Torch is only needed for train/serve.
TOKEN=os.environ.get('HF_TOKEN','')
if not TOKEN:
    try:
        from google.colab import userdata
        TOKEN=userdata.get('HF_TOKEN')
    except Exception:
        try:
            from kaggle_secrets import UserSecretsClient
            TOKEN=UserSecretsClient().get_secret('NEBULA_HF_TOKEN')
        except Exception:
            pass
# Never print the token. Public inventory can run without it.


In [ ]:
SOURCES={'vd2_data.py': '"""V-D2 data admission, provenance, deduplication and holdout isolation. No torch dependency."""\nimport hashlib,json,re,io,gzip,struct,unicodedata\nSECRET=re.compile(r\'\\b(?:hf_|gsk_|sk-|AIza)[A-Za-z0-9_-]{18,}|-----BEGIN [A-Z ]*PRIVATE KEY-----|\\bBearer\\s+[A-Za-z0-9._-]{20,}\')\nSPECIAL=[\'<|n27_pad|>\',\'<|n27_bos|>\',\'<|n27_eos|>\',\'<|n27_system|>\',\'<|n27_user|>\',\'<|n27_assistant|>\',\'<|fim_prefix|>\',\'<|fim_suffix|>\',\'<|fim_middle|>\']\ndef sha(b):return hashlib.sha256(b).hexdigest()\n\ndef source_split(source):\n    # Historical manifests use camelCase names, including publicHoldout.\n    names=\' \'.join([str(source.get(\'key\',\'\')),str(source.get(\'state\',\'\')),str(source.get(\'entry\',{}).get(\'path\',\'\'))]+[str(p.get(\'path\',\'\')) for p in source.get(\'entry\',{}).get(\'parts\',[])]).lower()\n    return \'validation\' if source.get(\'split\')==\'validation\' or any(x in names for x in [\'validation\',\'heldout\',\'held-out\',\'held_out\',\'holdout\',\'hold-out\',\'hold_out\']) else \'train\'\ndef parameter_count(vocab=8192,width=1536,layers=35,context=2048):return vocab*width+context*width+layers*(12*width*width+13*width)+2*width\n\ndef pointers(value,trail=()):\n    if isinstance(value,dict):\n        if type(value.get(\'bytes\')) is int and isinstance(value.get(\'sha256\'),str) and (\'path\' in value or \'parts\' in value):yield trail,value\n        else:\n            for k,v in value.items():yield from pointers(v,trail+(str(k),))\n    elif isinstance(value,list):\n        for i,v in enumerate(value):yield from pointers(v,trail+(str(i),))\n\ndef candidates(repo,revision,state_path,state):\n    out=[]\n    for keys,entry in pointers(state):\n        name=\'/\'.join(keys);path=entry.get(\'path\') or entry.get(\'parts\',[{}])[0].get(\'path\',\'\');s=(name+\' \'+path).lower()\n        if any(x in s for x in [\'eval\',\'probe\',\'checks\',\'judge\',\'rules\',\'report\',\'card-\',\'license\',\'index\',\'sourcecard\',\'baseline\']):continue\n        kind=\'jsonl\' if \'.jsonl\' in path else \'bpe-u16\' if name in [\'trainTokens\',\'validationTokens\'] else \'bytes-u16\' if name.startswith(\'corpus/train/\') or name.startswith(\'corpus/validation/\') else None\n        if not kind:continue\n        source=dict(repo=repo,revision=revision,state=state_path,key=name,entry=entry,format=kind,tokenizer=state.get(\'tokenizer\') if kind==\'bpe-u16\' else None,approved=False,license=\'\',origin=\'historical-unreviewed\')\n        source[\'split\']=source_split(source);out.append(source)\n    return out\n\ndef texts(blob,kind,tokenizer=None):\n    if kind==\'jsonl\':\n        for line in blob.decode(\'utf-8\',errors=\'strict\').splitlines():\n            if not line.strip():continue\n            r=json.loads(line)\n            if not isinstance(r,dict):continue\n            if (isinstance(r.get(\'source\'),dict) and r[\'source\'].get(\'kind\')==\'user\') or r.get(\'origin\') in [\'user-chat\',\'review-feed\']:continue\n            if isinstance(r.get(\'text\'),str):yield r[\'text\']\n            elif isinstance(r.get(\'code\'),str):yield r[\'code\']\n            elif isinstance(r.get(\'messages\'),list):\n                # Preserve whole approved conversations as code documentation, not a new chat objective.\n                messages=r[\'messages\']\n                if all(isinstance(m,dict) and m.get(\'role\') in [\'system\',\'user\',\'assistant\'] and isinstance(m.get(\'content\'),str) for m in messages):\n                    yield \'\\n\\n\'.join(m[\'content\'] for m in messages)\n    elif kind in [\'bytes-u16\',\'bpe-u16\']:\n        if len(blob)%2:raise ValueError(\'Unaligned uint16 corpus\')\n        chunk=[]\n        for (token,) in struct.iter_unpack(\'<H\',blob):\n            boundary=token>=256 if kind==\'bytes-u16\' else token<6\n            if boundary:\n                if chunk:yield bytes(chunk).decode(\'utf-8\',errors=\'strict\') if kind==\'bytes-u16\' else tokenizer.decode(chunk,skip_special_tokens=True)\n                chunk=[]\n            else:chunk.append(token)\n        if chunk:yield bytes(chunk).decode(\'utf-8\',errors=\'strict\') if kind==\'bytes-u16\' else tokenizer.decode(chunk,skip_special_tokens=True)\n    else:raise ValueError(\'Unsupported encoding: \'+kind)\n\ndef clean(text):\n    text=unicodedata.normalize(\'NFC\',text.replace(\'\\r\\n\',\'\\n\')).strip()\n    if not 60<=len(text)<=150000 or SECRET.search(text) or \'\\ufffd\' in text or any(x in text for x in SPECIAL):return None\n    if any(ord(c)<32 and c not in \'\\n\\t\\r\' for c in text):return None\n    return text\n\ndef category(text):\n    if re.search(r\'(?m)^\\s*(?:def |class |import |from \\w+ import |function |export |const |let |public |package |#include|SELECT |CREATE TABLE)|```(?:python|py|javascript|js|typescript|ts|sql|rust|go|java|cpp|html)\',text):return \'code\'\n    if re.search(r\'\\b(?:unit test|regression test|SQL injection|authorization|authentication|parameterized|compiler|algorithm|data structure|HTTP|database|API|function|programming)\\b\',text,re.I):return \'docs\'\n    return \'general\'\n\nclass Corpus:\n    def __init__(self,max_bytes=256_000_000):self.records={};self.bytes=0;self.max_bytes=max_bytes;self.rejected={}\n    def add(self,text,source):\n        text=clean(text)\n        if text is None:self.rejected[\'invalid-or-secret\']=self.rejected.get(\'invalid-or-secret\',0)+1;return\n        if not source.get(\'approved\') or not source.get(\'license\') or source.get(\'origin\') in [\'user-chat\',\'review-feed\',\'historical-unreviewed\']:raise ValueError(\'Explicit provenance and license review required\')\n        fingerprint=sha(re.sub(r\'\\s+\',\' \',text).encode());split=\'validation\' if source_split(source)==\'validation\' or int(fingerprint[:8],16)%50==0 else \'train\'\n        if fingerprint in self.records:\n            if split==\'validation\':self.records[fingerprint][\'split\']=\'validation\'\n            return\n        size=len(text.encode())\n        if self.bytes+size>self.max_bytes:raise ValueError(\'RAM corpus budget reached; reduce source selection or increase deliberate budget\')\n        self.records[fingerprint]=dict(id=fingerprint,text=text,split=split,category=category(text),source={k:source[k] for k in [\'repo\',\'revision\',\'key\',\'license\',\'origin\']});self.bytes+=size\n    def summary(self):\n        counts={}\n        for r in self.records.values():k=r[\'split\']+\'/\'+r[\'category\'];counts[k]=counts.get(k,0)+1\n        return dict(documents=len(self.records),uniqueUtf8Bytes=self.bytes,counts=counts,rejected=self.rejected)\n    def payload(self):return \'\\n\'.join(json.dumps(r,ensure_ascii=False) for r in self.records.values()).encode()\n', 'vd2_hub.py': '"""Private, atomic HF artifacts: RAM buffers -> immutable parts -> manifest. No local model/data cache."""\nimport copy,hashlib,json,re,io\nfrom urllib.parse import quote\n\ndef sha(b):return hashlib.sha256(b).hexdigest()\nclass Hub:\n    def __init__(self,repo,token,run=\'nebula-vd2-1b-code\'):\n        import requests\n        from huggingface_hub import HfApi\n        if not re.fullmatch(r\'[\\w-]+/[\\w.-]+\',repo) or not re.fullmatch(r\'nebula-vd2-[\\w-]{1,50}\',run):raise ValueError(\'Invalid output namespace\')\n        self.repo=repo;self.token=token;self.api=HfApi(token=token);self.session=requests.Session();self.prefix=\'experiments/\'+run+\'/\'\n        if self.api.whoami()[\'name\']!=repo.split(\'/\')[0]:raise ValueError(\'Output must belong to the token owner\')\n        self.api.create_repo(repo,repo_type=\'model\',private=True,exist_ok=True)\n        info=self.api.model_info(repo)\n        if not info.private:raise ValueError(\'Owner-only weights require a PRIVATE output repository; source repositories remain unchanged\')\n        self.revision=info.sha\n    def fetch(self,repo,path,revision,max_bytes=160000000):\n        if not re.fullmatch(r\'[\\w-]+/[\\w.-]+\',repo) or \'..\' in path.split(\'/\') or path.startswith(\'/\'):raise ValueError(\'Invalid source path\')\n        url=f\'https://huggingface.co/{repo}/resolve/{quote(revision,safe="")}/{quote(path,safe="/")}\'\n        with self.session.get(url,headers={\'Authorization\':\'Bearer \'+self.token},timeout=(30,180),stream=True) as r:\n            r.raise_for_status();out=bytearray()\n            for chunk in r.iter_content(1024*1024):\n                out.extend(chunk)\n                if len(out)>max_bytes:raise ValueError(\'Object exceeds declared RAM limit\')\n            return bytes(out)\n    def read_entry(self,repo,revision,entry):\n        if not 0<entry[\'bytes\']<=20_000_000_000:raise ValueError(\'Invalid object size\')\n        out=io.BytesIO()\n        for p in entry.get(\'parts\',[entry]):\n            data=self.fetch(repo,p[\'path\'],revision,max_bytes=p[\'bytes\'])\n            if len(data)!=p[\'bytes\'] or sha(data)!=p[\'sha256\']:raise ValueError(\'Source checksum mismatch\')\n            out.write(data)\n        data=out.getvalue()\n        if len(data)!=entry[\'bytes\'] or sha(data)!=entry[\'sha256\']:raise ValueError(\'Combined checksum mismatch\')\n        return data\n    def state(self):\n        import requests\n        try:return json.loads(self.fetch(self.repo,self.prefix+\'state.json\',self.revision,2_000_000))\n        except requests.HTTPError as e:\n            if e.response.status_code==404:return None\n            raise\n    def publish(self,state,artifacts):\n        from huggingface_hub import CommitOperationAdd\n        next=copy.deepcopy(state);ops=[]\n        for key,value in artifacts.items():\n            view=value.getbuffer() if isinstance(value,io.BytesIO) else memoryview(value)\n            digest=hashlib.sha256(view).hexdigest();parts=[]\n            for i,start in enumerate(range(0,len(view),64*1024*1024)):\n                b=bytes(view[start:start+64*1024*1024]);path=self.prefix+\'objects/\'+digest+f\'/part-{i:04d}.bin\'\n                parts.append(dict(path=path,bytes=len(b),sha256=sha(b)));ops.append(CommitOperationAdd(path_in_repo=path,path_or_fileobj=b))\n            next[key]=dict(bytes=len(view),sha256=digest,parts=parts)\n        ops.append(CommitOperationAdd(path_in_repo=self.prefix+\'state.json\',path_or_fileobj=json.dumps(next,indent=2).encode()))\n        commit=self.api.create_commit(repo_id=self.repo,parent_commit=self.revision,operations=ops,commit_message=\'V-D2 owner-only data/checkpoint backup\')\n        self.revision=commit.oid\n        if json.loads(self.fetch(self.repo,self.prefix+\'state.json\',self.revision,2_000_000))!=next:raise ValueError(\'Backup verification failed\')\n        print(json.dumps(dict(vd2BackupConfirmed=True,revision=self.revision,updates=next.get(\'updates\',0),private=True,websiteUpdated=False)),flush=True)\n        return next\n', 'vd2_lab.py': '"""Coding-focused consolidation and conservative 1B training. Artifacts persist only in private HF."""\nimport copy,gzip,hashlib,io,json,math,os,random,time\nfrom pathlib import Path\nfrom vd2_data import candidates,Corpus,texts,sha,SPECIAL,parameter_count,source_split\nfrom vd2_hub import Hub\nMODEL=dict(vocab=8192,width=1536,heads=24,layers=35,context=2048)\nDEFAULT=dict(model=MODEL,updates=10000,batch=8,microbatch=1,learningRate=0.0001,warmup=200,saveEvery=100,evaluateEvery=500,seed=31025,maxCorpusBytes=256_000_000,minCodeDocuments=5000)\n\ndef engine_hash():\n    root=Path(__file__).parent\n    names=[\'vd2_data.py\',\'vd2_hub.py\',\'vd2_lab.py\',\'vd2_fsdp.py\',\'v30_expanded_model.py\',\'v30_device_audit.py\']\n    return sha(\'\'.join((root/n).read_text(encoding=\'utf-8\') for n in names).encode())\n\ndef inventory(repos,token=None):\n    from huggingface_hub import HfApi\n    import requests\n    api=HfApi(token=token);entries=[];runs=[]\n    for repo in repos:\n        info=api.model_info(repo);revision=info.sha\n        for name in api.list_repo_files(repo,revision=revision):\n            if not name.startswith(\'experiments/\') or not name.endswith(\'/state.json\'):continue\n            url=f\'https://huggingface.co/{repo}/resolve/{revision}/{name}\'\n            r=requests.get(url,headers={\'Authorization\':\'Bearer \'+token} if token else {},timeout=60);r.raise_for_status()\n            if len(r.content)>2_000_000:raise ValueError(\'Manifest too large\')\n            state=r.json();runs.append(dict(repo=repo,revision=revision,path=name));entries+=candidates(repo,revision,name,state)\n    # Content duplicates share a candidate; source references remain recorded in the inventory.\n    return dict(format=\'nebula-vd2-inventory-1\',runs=runs,candidates=entries,notScanned=[\'Unconfigured GCS/S3 buckets\',\'Private repositories not accessible to supplied token\'],noObjectsMoved=True)\n\ndef prepare(repo,token,approved,config=DEFAULT):\n    from tokenizers import Tokenizer,models,pre_tokenizers,decoders,trainers\n    hub=Hub(repo,token);existing=hub.state()\n    fingerprint=sha(json.dumps({\'config\':config,\'sources\':approved,\'engine\':engine_hash()},sort_keys=True).encode())\n    if existing:\n        if existing.get(\'signature\')!=fingerprint:raise ValueError(\'Changed source/config: choose a new output repository/run, do not overwrite prior work\')\n        if existing.get(\'dataReady\'):return existing\n    if not approved or any(not x.get(\'approved\') or not x.get(\'license\') or x.get(\'origin\') in [\'historical-unreviewed\',\'user-chat\',\'review-feed\'] for x in approved):raise ValueError(\'Review and approve explicit training provenance/licenses first; do not admit private chats\')\n    corpus=Corpus(config[\'maxCorpusBytes\'])\n    # Process historical validation first. Dedup moves any duplicated training record to validation.\n    for source in sorted(approved,key=lambda x:source_split(x)!=\'validation\'):\n        blob=hub.read_entry(source[\'repo\'],source[\'revision\'],source[\'entry\']);tokenizer=None\n        if source[\'format\']==\'bpe-u16\':\n            if not source.get(\'tokenizer\'):raise ValueError(\'Cannot decode subwords without their exact tokenizer\')\n            tokenizer=Tokenizer.from_str(hub.read_entry(source[\'repo\'],source[\'revision\'],source[\'tokenizer\']).decode())\n        for text in texts(blob,source[\'format\'],tokenizer):corpus.add(text,source)\n        print(json.dumps({\'vd2Consolidating\':source[\'key\'],**corpus.summary()}),flush=True)\n        del blob\n    if corpus.summary()[\'counts\'].get(\'train/code\',0)<config[\'minCodeDocuments\']:raise ValueError(\'Not enough distinct code documents for this coding experiment. Historical story/chat data is not an adequate coding corpus.\')\n    tokenizer=Tokenizer(models.BPE(unk_token=None));tokenizer.pre_tokenizer=pre_tokenizers.ByteLevel(add_prefix_space=False);tokenizer.decoder=decoders.ByteLevel()\n    tokenizer.train_from_iterator((r[\'text\'] for r in corpus.records.values() if r[\'split\']==\'train\' and r[\'category\'] in [\'code\',\'docs\']),trainers.BpeTrainer(vocab_size=config[\'model\'][\'vocab\'],special_tokens=SPECIAL,initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))\n    if tokenizer.get_vocab_size()!=config[\'model\'][\'vocab\']:raise ValueError(\'Corpus too small to build the target vocabulary\')\n    state=dict(format=\'nebula-vd2-code-1\',signature=fingerprint,engine=engine_hash(),config=config,dataReady=True,updates=0,qualityApproved=False,websiteUpdated=False,summary=corpus.summary(),sources=approved)\n    return hub.publish(state,{\'corpus\':gzip.compress(corpus.payload()),\'tokenizer\':tokenizer.to_str().encode()})\n\ndef train(repo,token,hours=6.0,reserve_minutes=15,config=DEFAULT):\n    import numpy as np\n    import torch\n    import torch.nn.functional as F\n    from tokenizers import Tokenizer\n    from v30_expanded_model import Model,Config\n    from v30_device_audit import audit\n    # Full-parameter AdamW needs much more than 16GB. Two visible T4s do not pool memory automatically.\n    if not torch.cuda.is_available():raise ValueError(\'No CUDA GPU. Preparation/inventory can run on CPU; 1B training is not started.\')\n    props=torch.cuda.get_device_properties(0)\n    if props.total_memory<38*1024**3:raise ValueError(\'This implementation requires one >=40GB GPU for full 1B AdamW training. 2xT4 is not one pooled GPU. Use a larger GPU or implement/test sharded training separately; no purchase is made.\')\n    if hours<=reserve_minutes/60:raise ValueError(\'Reserve exceeds session duration\')\n    hub=Hub(repo,token);state=hub.state()\n    if not state or not state.get(\'dataReady\') or state[\'config\']!=config or state.get(\'engine\')!=engine_hash():raise ValueError(\'Prepare and freeze the approved coding corpus/config first\')\n    deadline=time.monotonic()+hours*3600-reserve_minutes*60\n    tokenizer_blob=hub.read_entry(repo,hub.revision,state[\'tokenizer\']);tokenizer=Tokenizer.from_str(tokenizer_blob.decode())\n    records=[json.loads(line) for line in gzip.decompress(hub.read_entry(repo,hub.revision,state[\'corpus\'])).decode().splitlines()]\n    pools={split:{kind:[] for kind in [\'code\',\'docs\',\'general\']} for split in [\'train\',\'validation\']}\n    for r in records:\n        ids=tokenizer.encode(r[\'text\'],add_special_tokens=False).ids+[2]\n        pools[r[\'split\']][r[\'category\']].append(ids)\n    if not pools[\'validation\'][\'code\']:raise ValueError(\'Independent coding validation documents required\')\n    torch.manual_seed(config[\'seed\']);random.seed(config[\'seed\']);model=Model(Config(**config[\'model\'])).cuda()\n    if sum(p.numel() for p in model.parameters())!=parameter_count(**{k:v for k,v in config[\'model\'].items() if k!=\'heads\'}):raise ValueError(\'Parameter count mismatch\')\n    bf16=torch.cuda.is_bf16_supported();dtype=torch.bfloat16 if bf16 else torch.float16\n    optimizer=torch.optim.AdamW([{\'params\':[p for p in model.parameters() if p.ndim>=2],\'weight_decay\':.1},{\'params\':[p for p in model.parameters() if p.ndim<2],\'weight_decay\':0}],lr=config[\'learningRate\'],betas=(.9,.95))\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=not bf16);step=state[\'updates\']\n    if state.get(\'working\'):\n        saved=torch.load(io.BytesIO(hub.read_entry(repo,hub.revision,state[\'working\'])),map_location=\'cpu\',weights_only=True)\n        if saved[\'signature\']!=state[\'signature\']:raise ValueError(\'Checkpoint configuration mismatch\')\n        model.load_state_dict(saved[\'model\']);optimizer.load_state_dict(saved[\'optimizer\']);scaler.load_state_dict(saved[\'scaler\']);random.setstate(saved[\'random\']);torch.set_rng_state(saved[\'torchRng\']);torch.cuda.set_rng_state(saved[\'cudaRng\']);del saved\n    # Same causal implementation as V30; evaluate/train agreement is checked before updates.\n    check_tokens=np.array((pools[\'train\'][\'code\'][0]*4)[:256],dtype=np.uint16)\n    report=audit(model,check_tokens)\n    if not report[\'passed\']:\n        model.set_attention_backend(\'math\');report=audit(model,check_tokens)\n        if not report[\'passed\']:raise ValueError(\'Device audit failed. No optimizer updates started.\')\n    state[\'deviceAudit\']=report\n    def example(split,seed):\n        rng=random.Random(seed);available=pools[split];kind=rng.choices([\'code\',\'docs\',\'general\'],[.75 if available[\'code\'] else 0,.2 if available[\'docs\'] else 0,.05 if available[\'general\'] else 0])[0]\n        data=[]\n        while len(data)<config[\'model\'][\'context\']+1:\n            ids=list(rng.choice(available[kind]))\n            if kind==\'code\' and len(ids)>64 and rng.random()<.2:\n                a,b=sorted(rng.sample(range(1,len(ids)-1),2));ids=[6]+ids[:a]+[7]+ids[b:]+[8]+ids[a:b]+[2]\n            data.extend(ids)\n        return torch.tensor(data[:config[\'model\'][\'context\']+1],device=\'cuda\',dtype=torch.long)\n    @torch.inference_mode()\n    def evaluate():\n        model.eval();losses=[]\n        for i in range(16):\n            ids=example(\'validation\',config[\'seed\']+900000+i)\n            with torch.autocast(\'cuda\',dtype=dtype):losses.append(float(F.cross_entropy(model(ids[:-1][None]).float().flatten(0,1),ids[1:])))\n        return dict(loss=sum(losses)/len(losses),targets=16*config[\'model\'][\'context\'],qualityApproved=False,notice=\'Held-out coding loss is not proof of correct or secure code. Run separate sandboxed tests before serving.\')\n    def save():\n        nonlocal state\n        buffer=io.BytesIO();torch.save(dict(format=\'nebula-vd2-code-1\',signature=state[\'signature\'],model={k:v.detach().cpu() for k,v in model.state_dict().items()},optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),random=random.getstate(),torchRng=torch.get_rng_state(),cudaRng=torch.cuda.get_rng_state()),buffer)\n        state=hub.publish({**state,\'updates\':step,\'supervisedTokens\':step*config[\'batch\']*config[\'model\'][\'context\']},{\'working\':buffer});buffer.close()\n    last=time.monotonic();last_saved=step\n    if not state.get(\'baseline\'):state=hub.publish({**state,\'baseline\':evaluate()}, {})\n    try:\n        while step<config[\'updates\'] and time.monotonic()<deadline:\n            model.train();optimizer.zero_grad(set_to_none=True);lr=config[\'learningRate\']*min(1,(step+1)/config[\'warmup\'])*(.1+.9*.5*(1+math.cos(math.pi*step/config[\'updates\'])))\n            for g in optimizer.param_groups:g[\'lr\']=lr\n            losses=[]\n            for micro in range(config[\'batch\']):\n                ids=example(\'train\',config[\'seed\']+step*config[\'batch\']+micro)\n                with torch.autocast(\'cuda\',dtype=dtype):loss=F.cross_entropy(model(ids[:-1][None]).float().flatten(0,1),ids[1:])/config[\'batch\']\n                if not torch.isfinite(loss):raise ValueError(\'Nonfinite loss; no successful checkpoint is replaced\')\n                scaler.scale(loss).backward();losses.append(float(loss.detach()))\n            scaler.unscale_(optimizer);torch.nn.utils.clip_grad_norm_(model.parameters(),1,error_if_nonfinite=True);scaler.step(optimizer);scaler.update();step+=1\n            if step%25==0:print(json.dumps(dict(vd2Update=step,parameters=parameter_count(),loss=sum(losses),lr=lr)),flush=True)\n            if step%config[\'evaluateEvery\']==0:state[\'evaluation\']=evaluate()\n            if step%config[\'saveEvery\']==0 or time.monotonic()-last>=180:\n                save();last_saved=step;last=time.monotonic()\n    except KeyboardInterrupt:print(\'Stop requested; saving this exact trajectory.\',flush=True)\n    # Upload errors propagate and stop the run; do not silently keep training without backups.\n    if last_saved!=step:save()\n    state=hub.publish({**state,\'evaluation\':evaluate()}, {})\n    return dict(updatesSaved=step,parameters=parameter_count(),complete=step>=config[\'updates\'],qualityApproved=False,websiteUpdated=False)\n', 'vd2_fsdp.py': '"""EXPERIMENTAL two-GPU FULL_SHARD trainer. Requires a real 2xT4 memory/resume smoke test.\nEach rank\'s optimizer is saved separately; only rank 0 atomically advances the model manifest.\nArtifacts remain in RAM -> private HF. Do not run two torchrun jobs for one output run.\n"""\nimport argparse,gc,gzip,io,json,os,random,time\nfrom datetime import timedelta\nfrom functools import partial\nfrom vd2_hub import Hub\nfrom vd2_lab import DEFAULT,engine_hash\n\ndef run(repo,token,hours=6,reserve_minutes=15,max_session_updates=0):\n    import numpy as np\n    import torch\n    import torch.distributed as dist\n    import torch.nn.functional as F\n    from tokenizers import Tokenizer\n    from torch.distributed.fsdp import FullyShardedDataParallel as FSDP,ShardingStrategy,MixedPrecision,StateDictType,FullStateDictConfig\n    from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy\n    from torch.distributed.fsdp.sharded_grad_scaler import ShardedGradScaler\n    from v30_expanded_model import Model,Config,Block\n    from v30_device_audit import audit\n    rank=int(os.environ.get(\'RANK\',-1));local=int(os.environ.get(\'LOCAL_RANK\',-1));world=int(os.environ.get(\'WORLD_SIZE\',0))\n    if world!=2 or not torch.cuda.is_available() or torch.cuda.device_count()<2:raise ValueError(\'Launch with torchrun --standalone --nproc_per_node=2 vd2_fsdp.py; requires two actual CUDA GPUs\')\n    torch.cuda.set_device(local);device=torch.device(\'cuda\',local);dist.init_process_group(\'nccl\',device_id=device,timeout=timedelta(minutes=30))\n    try:\n        hub=Hub(repo,token);state=hub.state()\n        if not state or not state.get(\'dataReady\') or state.get(\'engine\')!=engine_hash():raise ValueError(\'Prepare the frozen approved corpus first\')\n        config=state[\'config\'];step=state.get(\'updates\',0);session_target=min(config[\'updates\'],step+max_session_updates) if max_session_updates else config[\'updates\']\n        if state.get(\'worldSize\',2)!=2:raise ValueError(\'Cannot resume a single-GPU optimizer as two FSDP optimizers; use a separate deliberate weight-transfer experiment\')\n        if config[\'batch\']%2:raise ValueError(\'Global batch must divide by two ranks\')\n        if not 0<reserve_minutes/60<hours:raise ValueError(\'Invalid session/reserve\')\n        deadline=time.monotonic()+hours*3600-reserve_minutes*60\n        tokenizer=Tokenizer.from_str(hub.read_entry(repo,hub.revision,state[\'tokenizer\']).decode())\n        pools={s:{k:[] for k in [\'code\',\'docs\',\'general\']} for s in [\'train\',\'validation\']}\n        records=gzip.decompress(hub.read_entry(repo,hub.revision,state[\'corpus\'])).decode()\n        for line in records.splitlines():\n            r=json.loads(line);pools[r[\'split\']][r[\'category\']].append(tokenizer.encode(r[\'text\'],add_special_tokens=False).ids+[2])\n        del records;gc.collect()\n        if not pools[\'train\'][\'code\'] or not pools[\'validation\'][\'code\']:raise ValueError(\'Independent coding training and validation documents are required\')\n        torch.manual_seed(config[\'seed\']);base=Model(Config(**config[\'model\']))\n        if state.get(\'working\'):\n            blob=hub.read_entry(repo,hub.revision,state[\'working\']);saved=torch.load(io.BytesIO(blob),map_location=\'cpu\',weights_only=True)\n            if saved[\'signature\']!=state[\'signature\']:raise ValueError(\'Model signature mismatch\')\n            base.load_state_dict(saved[\'model\']);del blob,saved;gc.collect()\n        # Before optimizer allocation, audit the actual full model/device (small sequence).\n        base.to(device);probe=np.array((pools[\'train\'][\'code\'][0]*4)[:128],dtype=np.uint16);report=audit(base,probe)\n        if not report[\'passed\']:\n            base.set_attention_backend(\'math\');report=audit(base,probe)\n            if not report[\'passed\']:raise ValueError(\'Device audit failed before FSDP updates\')\n        if rank==0:print(json.dumps(dict(vd2FSDPAudit=report,experimental=True)),flush=True)\n        model=FSDP(base,auto_wrap_policy=partial(transformer_auto_wrap_policy,transformer_layer_cls={Block}),sharding_strategy=ShardingStrategy.FULL_SHARD,mixed_precision=MixedPrecision(param_dtype=torch.float16,reduce_dtype=torch.float32,buffer_dtype=torch.float16),device_id=device,use_orig_params=True,limit_all_gathers=True)\n        optimizer=torch.optim.AdamW([{\'params\':[p for n,p in model.named_parameters() if not any(s in n for s in [\'bias\',\'norm\'])],\'weight_decay\':.1},{\'params\':[p for n,p in model.named_parameters() if any(s in n for s in [\'bias\',\'norm\'])],\'weight_decay\':0}],lr=config[\'learningRate\'],betas=(.9,.95),foreach=False)\n        scaler=ShardedGradScaler(enabled=True)\n        opt_key=\'optimizer\'+str(rank)\n        if step:\n            if not state.get(opt_key):raise ValueError(\'Missing rank optimizer; refuse an incomplete resume\')\n            saved=torch.load(io.BytesIO(hub.read_entry(repo,hub.revision,state[opt_key])),map_location=\'cpu\',weights_only=True)\n            if saved[\'step\']!=step or saved[\'worldSize\']!=2 or saved[\'torchVersion\']!=str(torch.__version__):raise ValueError(\'Rank optimizer/version mismatch\')\n            optimizer.load_state_dict(saved[\'optimizer\']);scaler.load_state_dict(saved[\'scaler\']);random.setstate(saved[\'random\']);torch.set_rng_state(saved[\'torchRng\']);torch.cuda.set_rng_state(saved[\'cudaRng\'],device);del saved;gc.collect()\n        def example(split,seed):\n            rng=random.Random(seed);p=pools[split];kind=rng.choices([\'code\',\'docs\',\'general\'],[.75 if p[\'code\'] else 0,.2 if p[\'docs\'] else 0,.05 if p[\'general\'] else 0])[0];data=[]\n            while len(data)<config[\'model\'][\'context\']+1:\n                ids=list(rng.choice(p[kind]))\n                if split==\'train\' and kind==\'code\' and len(ids)>64 and rng.random()<.2:\n                    a,b=sorted(rng.sample(range(1,len(ids)-1),2));ids=[6]+ids[:a]+[7]+ids[b:]+[8]+ids[a:b]+[2]\n                data.extend(ids)\n            return torch.tensor(data[:config[\'model\'][\'context\']+1],device=device,dtype=torch.long)\n        @torch.inference_mode()\n        def evaluate():\n            model.eval();total=torch.zeros(1,device=device)\n            for i in range(4):\n                ids=example(\'validation\',config[\'seed\']+900000+rank*4+i)\n                with torch.autocast(\'cuda\',dtype=torch.float16):total+=F.cross_entropy(model(ids[:-1][None]).float().flatten(0,1),ids[1:])\n            dist.all_reduce(total);return dict(loss=float(total)/(4*world),targets=8*config[\'model\'][\'context\'],qualityApproved=False)\n        def save():\n            nonlocal state\n            # Collect weights only: no rank ever gathers the full 8GB optimizer state.\n            with FSDP.state_dict_type(model,StateDictType.FULL_STATE_DICT,FullStateDictConfig(offload_to_cpu=True,rank0_only=True)):weights=model.state_dict()\n            opt=io.BytesIO();torch.save(dict(step=step,worldSize=2,torchVersion=str(torch.__version__),optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),random=random.getstate(),torchRng=torch.get_rng_state(),cudaRng=torch.cuda.get_rng_state(device)),opt)\n            # Rank 1 uses a separate manifest. It cannot accidentally advance the main run\'s step.\n            metadata=[None]\n            if rank==1:\n                sibling=Hub(repo,token,run=\'nebula-vd2-1b-code-rank1\');s=sibling.publish({\'updates\':step,\'signature\':state[\'signature\']},{\'optimizer1\':opt});metadata[0]={\'revision\':sibling.revision,\'entry\':s[\'optimizer1\']}\n            dist.broadcast_object_list(metadata,src=1,device=device);opt1=metadata[0];opt.close() if rank==1 else None\n            if rank==0:\n                hub.revision=opt1[\'revision\'];model_blob=io.BytesIO();torch.save(dict(format=\'nebula-vd2-code-1\',signature=state[\'signature\'],model=weights),model_blob)\n                state=hub.publish({**state,\'updates\':step,\'worldSize\':2,\'optimizer1\':opt1[\'entry\'],\'supervisedTokens\':step*config[\'batch\']*config[\'model\'][\'context\'],\'experimentalFSDP\':True},{\'working\':model_blob,\'optimizer0\':opt});model_blob.close();opt.close()\n            del weights;gc.collect()\n            metadata=[{\'revision\':hub.revision,\'state\':state} if rank==0 else None];dist.broadcast_object_list(metadata,src=0,device=device);hub.revision=metadata[0][\'revision\'];state=metadata[0][\'state\'];dist.barrier(device_ids=[local])\n        if rank==0:print(json.dumps(dict(vd2FSDP=True,worldSize=2,precision=\'fp16\',fullShard=True,resumedUpdates=step,experimental=True)),flush=True)\n        state[\'evaluation\']=evaluate();last=time.monotonic();last_saved=step\n        while step<session_target:\n            keep=torch.tensor([time.monotonic()<deadline],device=device,dtype=torch.int32);dist.all_reduce(keep,op=dist.ReduceOp.MIN)\n            if not int(keep):break\n            model.train();optimizer.zero_grad(set_to_none=True)\n            import math\n            lr=config[\'learningRate\']*min(1,(step+1)/config[\'warmup\'])*(.1+.9*.5*(1+math.cos(math.pi*step/config[\'updates\'])))\n            for group in optimizer.param_groups:group[\'lr\']=lr\n            loss_sum=0.\n            # Do not use no_sync(): it keeps full accumulated gradients and can defeat memory savings.\n            for micro in range(config[\'batch\']//world):\n                ids=example(\'train\',config[\'seed\']+step*config[\'batch\']+rank*(config[\'batch\']//world)+micro)\n                with torch.autocast(\'cuda\',dtype=torch.float16):loss=F.cross_entropy(model(ids[:-1][None]).float().flatten(0,1),ids[1:])/(config[\'batch\']//world)\n                if not torch.isfinite(loss):raise ValueError(\'Nonfinite loss; previous confirmed snapshot remains intact\')\n                scaler.scale(loss).backward();loss_sum+=float(loss.detach())\n            scaler.unscale_(optimizer);model.clip_grad_norm_(1.);old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update()\n            if scaler.get_scale()<old_scale:\n                if rank==0:print(json.dumps(dict(vd2FSDPSkippedOverflow=True,scale=scaler.get_scale())),flush=True)\n                continue\n            step+=1\n            if rank==0 and step%25==0:print(json.dumps(dict(vd2FSDPUpdate=step,lossRank0=loss_sum,worldSize=2,peakRank0GpuBytes=torch.cuda.max_memory_allocated(device))),flush=True)\n            if step%config[\'evaluateEvery\']==0:state[\'evaluation\']=evaluate()\n            should_save=torch.tensor([step%config[\'saveEvery\']==0 or time.monotonic()-last>=180],device=device,dtype=torch.int32);dist.all_reduce(should_save,op=dist.ReduceOp.MAX)\n            if int(should_save):save();last_saved=step;last=time.monotonic()\n        if last_saved!=step:save()\n        state[\'evaluation\']=evaluate()\n        if rank==0:hub.publish(state,{})\n    finally:dist.destroy_process_group()\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--repo\',required=True);p.add_argument(\'--hours\',type=float,default=6);p.add_argument(\'--reserve-minutes\',type=float,default=15);p.add_argument(\'--max-session-updates\',type=int,default=0);args=p.parse_args();run(args.repo,os.environ[\'HF_TOKEN\'],args.hours,args.reserve_minutes,args.max_session_updates)\n', 'vd2_serve.py': '"""Private inference worker. No website crawling, execution or permission-grant tools here."""\nimport hmac,io,json,os,hashlib\n\ndef create_app():\n    import torch\n    from fastapi import FastAPI,Header,HTTPException,Request\n    from tokenizers import Tokenizer\n    from v30_expanded_model import Model,Config,encode_chat\n    from vd2_hub import Hub\n    # Request is looked up from globals by FastAPI on some releases.\n    globals()[\'Request\']=Request\n    token=os.environ.get(\'NEBULA_VD2_WORKER_TOKEN\',\'\')\n    if len(token)<32:raise RuntimeError(\'Configure a strong private worker token\')\n    hub=Hub(os.environ[\'NEBULA_VD2_REPO\'],os.environ[\'HF_TOKEN\']);state=hub.state()\n    if not state or not state.get(\'working\'):raise RuntimeError(\'No trained V-D2 weights exist\')\n    if not state.get(\'qualityApproved\') and os.environ.get(\'NEBULA_VD2_ALLOW_EXPERIMENTAL\')!=\'true\':raise RuntimeError(\'Unapproved checkpoint. Complete coding evaluation or explicitly enable owner-only experimental serving.\')\n    blob=hub.read_entry(hub.repo,hub.revision,state[\'working\']);saved=torch.load(io.BytesIO(blob),map_location=\'cpu\',weights_only=True)\n    model=Model(Config(**state[\'config\'][\'model\']));model.load_state_dict(saved[\'model\']);del saved,blob\n    device=\'cuda\' if torch.cuda.is_available() else \'cpu\';model.to(device).eval();tokenizer=Tokenizer.from_str(hub.read_entry(hub.repo,hub.revision,state[\'tokenizer\']).decode());checkpoint=\'sha256:\'+state[\'working\'][\'sha256\']\n    import asyncio\n    lock=asyncio.Lock();app=FastAPI(docs_url=None,redoc_url=None,openapi_url=None)\n    @app.post(\'/generate\')\n    async def generate(request:Request,authorization:str=Header(default=\'\')):\n        if not hmac.compare_digest(authorization,\'Bearer \'+token):raise HTTPException(401,\'Unauthorized\')\n        body=bytearray()\n        async for chunk in request.stream():\n            body.extend(chunk)\n            if len(body)>120000:raise HTTPException(413,\'Request too large\')\n        try:\n            value=json.loads(body);messages=value[\'messages\'];limit=value[\'maxNewTokens\']\n            if value[\'model\']!=\'nebula-v-d2\' or len(messages)!=2 or messages[0][\'role\']!=\'system\' or messages[1][\'role\']!=\'user\' or type(limit)is not int or not 1<=limit<=2048:raise ValueError()\n            ids,_=encode_chat(tokenizer,messages,answer_prefix=True)\n        except Exception:raise HTTPException(400,\'Invalid generation request\')\n        if len(ids)>=model.config.context-32:raise HTTPException(413,\'Prompt exceeds context; send a smaller code excerpt\')\n        if lock.locked():raise HTTPException(429,\'Worker busy\')\n        def complete():\n            out=[]\n            with torch.inference_mode():\n                for _ in range(min(limit,model.config.context-len(ids))):\n                    x=torch.tensor([ids+out],device=device)\n                    logits=model(x)[0,-1].float();logits[[0,1,3,4,5,6,7,8]]=-float(\'inf\');next=int(logits.argmax())\n                    if next==2:break\n                    out.append(next)\n            return tokenizer.decode(out,skip_special_tokens=True)\n        async with lock:text=await asyncio.to_thread(complete)\n        return dict(model=\'nebula-v-d2\',checkpoint=checkpoint,text=text,experimental=not state.get(\'qualityApproved\',False))\n    return app\n', 'v30_expanded_model.py': '"""Depth-expanded subword model with explicit causal attention and inherited weights."""\nimport io\nimport math\nfrom dataclasses import dataclass,asdict\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom torch.utils.checkpoint import checkpoint as activation_checkpoint\n\nSPECIAL=[\'<|n27_pad|>\',\'<|n27_bos|>\',\'<|n27_eos|>\',\'<|n27_system|>\',\'<|n27_user|>\',\'<|n27_assistant|>\']\nPAD,BOS,EOS,SYSTEM,USER,ASSISTANT=range(6)\nROLES={\'system\':SYSTEM,\'user\':USER,\'assistant\':ASSISTANT}\n@dataclass\nclass Config:\n    vocab:int=8192\n    width:int=768\n    heads:int=12\n    layers:int=34\n    context:int=1024\n\nclass Block(nn.TransformerEncoderLayer):\n    # Keep legacy parameter names, but bypass eval-only fused encoder/MHA paths.\n    def forward(self,x,src_mask=None):\n        z=self.norm1(x);b,n,w=z.shape;h=self.self_attn.num_heads;d=w//h\n        q,k,v=F.linear(z,self.self_attn.in_proj_weight,self.self_attn.in_proj_bias).chunk(3,dim=-1)\n        q,k,v=[t.reshape(b,n,h,d).transpose(1,2) for t in (q,k,v)]\n        if getattr(self,\'attention_backend\',\'default\')==\'math\':\n            from torch.nn.attention import sdpa_kernel,SDPBackend\n            with sdpa_kernel(SDPBackend.MATH):\n                a=F.scaled_dot_product_attention(q,k,v,dropout_p=0.,is_causal=True)\n        else:a=F.scaled_dot_product_attention(q,k,v,dropout_p=0.,is_causal=True)\n        a=a.transpose(1,2).contiguous().reshape(b,n,w)\n        x=x+self.self_attn.out_proj(a)\n        return x+self.linear2(F.gelu(self.linear1(self.norm2(x))))\n\nclass Model(nn.Module):\n    def __init__(self,config):\n        super().__init__();self.config=config\n        self.token=nn.Embedding(config.vocab,config.width)\n        self.position=nn.Embedding(config.context,config.width)\n        self.blocks=nn.ModuleList(Block(config.width,config.heads,config.width*4,\n            dropout=0.,activation=\'gelu\',batch_first=True,norm_first=True) for _ in range(config.layers))\n        self.norm=nn.LayerNorm(config.width)\n        self.output=nn.Linear(config.width,config.vocab,bias=False);self.output.weight=self.token.weight\n        self.register_buffer(\'mask\',torch.ones(config.context,config.context,dtype=torch.bool).triu(1),persistent=False)\n        self.gradient_checkpointing=True\n        self.apply(self.initialize)\n        for block in self.blocks:\n            nn.init.normal_(block.self_attn.out_proj.weight,std=.02/math.sqrt(2*config.layers))\n            nn.init.normal_(block.linear2.weight,std=.02/math.sqrt(2*config.layers))\n    def set_attention_backend(self,backend):\n        if backend not in [\'default\',\'math\']:raise ValueError(\'Unsupported attention backend\')\n        for block in self.blocks:block.attention_backend=backend\n\n    @staticmethod\n    def initialize(module):\n        if isinstance(module,(nn.Linear,nn.Embedding)):\n            nn.init.normal_(module.weight,std=.02)\n            if getattr(module,\'bias\',None) is not None:nn.init.zeros_(module.bias)\n    def forward(self,tokens):\n        n=tokens.shape[1]\n        if n>self.config.context:raise ValueError(\'Context exceeded\')\n        x=self.token(tokens)+self.position(torch.arange(n,device=tokens.device))\n        for block in self.blocks:\n            if self.training and torch.is_grad_enabled() and self.gradient_checkpointing:\n                x=activation_checkpoint(block,x,use_reentrant=False)\n            else:x=block(x)\n        return self.output(self.norm(x))\n\n\ndef encode_chat(tokenizer,messages,answer_prefix=False):\n    tokens=[BOS];learn=[False]\n    for message in messages:\n        role=message[\'role\'];text=message[\'content\']\n        if role not in ROLES or any(s in text for s in SPECIAL):raise ValueError(\'Invalid chat role or reserved marker\')\n        ids=tokenizer.encode(text,add_special_tokens=False).ids\n        tokens+=[ROLES[role]]+ids+[EOS]\n        learn+=[False]+[role==\'assistant\']*(len(ids)+1)\n    if answer_prefix:tokens+=[ASSISTANT];learn+=[False]\n    return tokens,learn\n\n\ndef chat_window(tokenizer,messages,context):\n    tokens,learn=encode_chat(tokenizer,messages)\n    if len(tokens)>context+1:raise ValueError(\'Complete conversation does not fit\')\n    x=torch.tensor(tokens[:-1],dtype=torch.long)\n    y=torch.tensor([v if flag else -100 for v,flag in zip(tokens[1:],learn[1:])],dtype=torch.long)\n    if not (y!=-100).any():raise ValueError(\'No assistant targets\')\n    return x,y\n\n\ndef checkpoint(model,training,tokenizer_sha):\n    if any(not torch.isfinite(p).all() for p in model.parameters()):raise ValueError(\'Nonfinite checkpoint\')\n    stream=io.BytesIO()\n    torch.save({\'format\':\'nebula-v30-expanded-subword-1\',\'config\':asdict(model.config),\'model\':model.state_dict(),\n        \'tokenizerSha256\':tokenizer_sha,\'training\':training},stream)\n    return stream\n\n\ndef restore(blob,tokenizer_sha):\n    value=torch.load(io.BytesIO(blob),map_location=\'cpu\',weights_only=True)\n    if value.get(\'format\')!=\'nebula-v30-expanded-subword-1\' or value[\'tokenizerSha256\']!=tokenizer_sha:\n        raise ValueError(\'Incompatible model/tokenizer; use the V30 checkpoint format\')\n    model=Model(Config(**value[\'config\']));model.load_state_dict(value[\'model\'])\n    return model,value[\'training\']\n\n\n@torch.inference_mode()\ndef generate(model,tokenizer,prompt,chat=False,limit=96):\n    model.eval()\n    ids=encode_chat(tokenizer,[{\'role\':\'user\',\'content\':prompt}],True)[0] if chat else [BOS]+tokenizer.encode(prompt).ids\n    if len(ids)>=model.config.context:raise ValueError(\'Probe too long\')\n    device=next(model.parameters()).device;out=[]\n    for _ in range(min(limit,model.config.context-len(ids))):\n        x=torch.tensor([ids+out],device=device)\n        logits=model(x)[0,-1].float();logits[[PAD,BOS,SYSTEM,USER,ASSISTANT]]=-float(\'inf\')\n        token=int(logits.argmax())\n        if token==EOS:break\n        out.append(token)\n    return tokenizer.decode(out,skip_special_tokens=True)\n\n\ndef expand_parent(blob,tokenizer_sha,target):\n    value=torch.load(io.BytesIO(blob),map_location=\'cpu\',weights_only=True)\n    if value.get(\'format\')!=\'nebula-v27-subword-1\' or value[\'tokenizerSha256\']!=tokenizer_sha:\n        raise ValueError(\'Expected original V28 subword weights and exact tokenizer\')\n    original=Config(**value[\'config\'])\n    for key in [\'vocab\',\'width\',\'heads\',\'context\']:\n        if getattr(original,key)!=getattr(target,key):raise ValueError(\'Depth expansion cannot change \'+key)\n    if target.layers<=original.layers:raise ValueError(\'Expansion must add layers\')\n    model=Model(target);weights=model.state_dict()\n    for key,tensor in value[\'model\'].items():\n        if key not in weights or weights[key].shape!=tensor.shape:raise ValueError(\'Legacy parameter mismatch: \'+key)\n        weights[key]=tensor\n    model.load_state_dict(weights)\n    with torch.no_grad():\n        for block in model.blocks[original.layers:]:\n            block.self_attn.out_proj.weight.zero_();block.self_attn.out_proj.bias.zero_()\n            block.linear2.weight.zero_();block.linear2.bias.zero_()\n    return model,dict(originalLayers=original.layers,expandedLayers=target.layers,\n        inheritedWeights=True,newResidualOutputsInitiallyZero=True,parentOptimizerDiscarded=True,\n        parameters=sum(p.numel() for p in model.parameters()))\n', 'v30_device_audit.py': '"""Check actual device/model masking and train/eval consistency before updates."""\nimport numpy as np\nimport torch\n\n@torch.inference_mode()\ndef audit(model,values):\n    original_mode=model.training;tf32=torch.backends.cuda.matmul.allow_tf32\n    device=next(model.parameters()).device;checks={};errors={}\n    try:\n        torch.backends.cuda.matmul.allow_tf32=False\n        n=min(64,model.config.context,len(values)-1)\n        seq=torch.tensor(np.array(values[:n],dtype=np.int64),device=device)[None]\n        target=torch.tensor(np.array(values[1:n+1],dtype=np.int64),device=device)[None]\n        checks[\'one_token_shift\']=torch.equal(seq[:,1:],target[:,:-1])\n        model.eval();full=model(seq).float()\n        model.train()\n        with torch.inference_mode(False),torch.enable_grad():\n            training=model(seq.clone()).float().detach()\n        errors[\'trainEvalMaxAbs\']=float((full-training).abs().max())\n        checks[\'train_eval_agree\']=torch.allclose(full,training,atol=1e-3,rtol=1e-4)\n        model.eval();cut=n//2;changed=seq.clone();changed[:,cut:]=(changed[:,cut:]+7)%model.config.vocab\n        altered=model(changed).float()\n        errors[\'futureMaxAbs\']=float((full[:,:cut]-altered[:,:cut]).abs().max())\n        checks[\'future_cannot_change_past\']=torch.allclose(full[:,:cut],altered[:,:cut],atol=1e-3,rtol=1e-4)\n        prefix=model(seq[:,:cut]).float()\n        errors[\'prefixMaxAbs\']=float((full[:,cut-1]-prefix[:,-1]).abs().max())\n        checks[\'prefix_and_teacher_forced_agree\']=torch.allclose(full[:,cut-1],prefix[:,-1],atol=1e-3,rtol=1e-4)\n        checks[\'finite_logits\']=bool(torch.isfinite(full).all())\n        return dict(format=\'nebula-v30-device-audit-1\',passed=all(checks.values()),checks=checks,maxErrors=errors,\n            device=str(device),scope=\'Small actual checkpoint/device sample; does not certify factual knowledge or all training behavior.\')\n    finally:\n        model.train(original_mode);torch.backends.cuda.matmul.allow_tf32=tf32\n', 'vd2_audit.py': '"""Read-only, bounded data inspection. Reports counts, never text or approvals."""\nimport collections,json,re\nfrom vd2_data import texts,clean,category,sha,source_split\nfrom vd2_hub import Hub\n\ndef audit_sources(inventory,token=None,max_bytes=512_000_000,max_object_bytes=128_000_000,reader=None):\n    if reader is None:\n        import requests\n        hub=object.__new__(Hub);hub.token=token or \'\';hub.session=requests.Session()\n        reader=hub.read_entry # Does not construct Hub, create a repo or write anything.\n    groups={}\n    for source in inventory[\'candidates\']:\n        groups.setdefault((source[\'entry\'][\'sha256\'],source[\'format\']),[]).append(source)\n    records={};reports=[];used=0\n    for aliases in sorted(groups.values(),key=lambda a:all(source_split(s)!=\'validation\' for s in a)):\n        source=aliases[0];size=source[\'entry\'][\'bytes\']\n        split=\'validation\' if any(source_split(s)==\'validation\' for s in aliases) else \'train\'\n        report=dict(repo=source[\'repo\'],revision=source[\'revision\'],key=source[\'key\'],aliases=len(aliases),split=split,bytes=size,approved=False)\n        if source[\'format\']!=\'jsonl\':report[\'status\']=\'not_inspected_encoded_tokens\'\n        elif size>max_object_bytes or used+size>max_bytes:report[\'status\']=\'not_inspected_byte_budget\'\n        else:\n            used+=size\n            try:\n                blob=reader(source[\'repo\'],source[\'revision\'],source[\'entry\'])\n                counts=collections.Counter();rejected=0;pending={}\n                for value in texts(blob,\'jsonl\'):\n                    text=clean(value)\n                    if text is None:rejected+=1;continue\n                    fingerprint=sha(re.sub(r\'\\s+\',\' \',text).encode());kind=category(text);counts[kind]+=1\n                    target=\'validation\' if split==\'validation\' or int(fingerprint[:8],16)%50==0 else \'train\'\n                    if fingerprint not in pending or target==\'validation\':pending[fingerprint]=(target,kind)\n                # A malformed object must not contribute a partially parsed corpus.\n                for fingerprint,value in pending.items():\n                    if fingerprint not in records or value[0]==\'validation\':records[fingerprint]=value\n                report.update(status=\'inspected\',counts=dict(counts),rejected=rejected)\n                del blob\n            except Exception as error:\n                # Exception messages may contain signed URLs or example text.\n                report.update(status=\'inspection_failed\',errorType=type(error).__name__)\n        reports.append(report)\n        print(json.dumps({\'vd2SourceAudit\':report}),flush=True)\n    totals=collections.Counter(\'/\'.join(v) for v in records.values())\n    result=dict(readOnly=True,objectsMoved=False,trainingStarted=False,sourcesApproved=False,bytesRead=used,\n                inspectedObjects=sum(r[\'status\']==\'inspected\' for r in reports),totalObjects=len(reports),\n                uniqueInspectedDocuments=len(records),counts=dict(totals),minimumCodeDocuments=5000,\n                inspectedCodeFloorMet=totals[\'train/code\']>=5000,\n                note=\'Code counts are heuristic; provenance, licenses and uninspected objects still require review.\',sources=reports)\n    print(json.dumps({\'vd2AuditSummary\':{k:v for k,v in result.items() if k!=\'sources\'}}),flush=True)\n    return result\n'}
code_root=Path(tempfile.gettempdir())/'nebula-vd2-code'
code_root.mkdir(exist_ok=True)
for name,source in SOURCES.items():(code_root/name).write_text(source,encoding='utf-8')
sys.path.insert(0,str(code_root))
for name in SOURCES:sys.modules.pop(Path(name).stem,None)
from vd2_lab import inventory,prepare,train,DEFAULT
from vd2_data import parameter_count
print({'preparedParameters':parameter_count(),'trainingStarted':False,'websiteUpdated':False})


In [ ]:
import copy
PHASE='audit' # audit (read-only), inventory, prepare, train
TRAINER='two-t4' # experimental full-shard, or single-large-gpu
RUN_SMOKE_FIRST=True
SOURCE_REPOS=['iiidevv/nebula-research','iiidevv/nebula-raw-training']
OUTPUT_REPO='iiidevv/nebula-vd2-private' # NEW/PRIVATE, owned by the HF token account
CONFIG=copy.deepcopy(DEFAULT)
SESSION_HOURS=6.0
RESERVE_MINUTES=15
# Reviewed copies of inventory candidates. Each needs approved=True, license and accurate origin.
# Example origin: public-licensed or owner-code. Never user-chat or review-feed.
APPROVED_SOURCES=[]


In [ ]:
if PHASE in ['inventory','audit']:
    INVENTORY=inventory(SOURCE_REPOS,TOKEN or None)
    print({'runs':len(INVENTORY['runs']),'candidates':len(INVENTORY['candidates']),'objectsMoved':False})
    for i,s in enumerate(INVENTORY['candidates']):print(i,s['repo'],s['key'],s['format'],s['split'],s['entry']['bytes'])
    if PHASE=='audit':
        from vd2_audit import audit_sources
        AUDIT=audit_sources(INVENTORY,TOKEN or None)
        print('Audit complete. No data approved, copied or trained. Review counts before preparing a coding corpus.')
elif PHASE=='prepare':
    if not TOKEN:raise RuntimeError('Add your HF write token to Secrets, never chat/output.')
    result=prepare(OUTPUT_REPO,TOKEN,APPROVED_SOURCES,CONFIG)
    print({'dataReady':result['dataReady'],'summary':result['summary'],'trainingStarted':False})
elif PHASE=='train':
    if not TOKEN:raise RuntimeError('Configure HF_TOKEN in Secrets.')
    if TRAINER=='two-t4':
        import torch,time,socket
        from vd2_hub import Hub
        if torch.cuda.device_count()<2:raise RuntimeError('Enable two actual CUDA GPUs.')
        state=Hub(OUTPUT_REPO,TOKEN).state()
        if not state or not state.get('dataReady'):raise RuntimeError('Prepare reviewed coding sources first.')
        started=time.monotonic()
        env={**os.environ,'HF_TOKEN':TOKEN,'PYTHONPATH':os.pathsep.join([str(code_root),str(packages),os.environ.get('PYTHONPATH','')])}
        with socket.socket() as rendezvous:
            rendezvous.bind(('127.0.0.1',0));port=rendezvous.getsockname()[1]
        command=[sys.executable,'-m','torch.distributed.run','--nnodes=1','--master_addr=127.0.0.1','--master_port='+str(port),'--nproc_per_node=2',str(code_root/'vd2_fsdp.py'),'--repo',OUTPUT_REPO,'--reserve-minutes',str(RESERVE_MINUTES)]
        if RUN_SMOKE_FIRST and not state.get('updates'):
            print('Running two-update sharded save check; private uploads may take minutes.')
            subprocess.run(command+['--hours',str(SESSION_HOURS),'--max-session-updates','2'],env=env,check=True)
        remaining=SESSION_HOURS-(time.monotonic()-started)/3600
        if remaining>RESERVE_MINUTES/60:
            print('Resuming the exact two-GPU checkpoint/optimizer for the remaining session.')
            subprocess.run(command+['--hours',str(remaining)],env=env,check=True)
        else:print('Time reserve reached. Resume the confirmed backup later.')
    elif TRAINER=='single-large-gpu':
        print(train(OUTPUT_REPO,TOKEN,SESSION_HOURS,RESERVE_MINUTES,CONFIG))
    else:raise ValueError('Unknown trainer')
else:raise ValueError('Unknown phase')
